# Workshop Prático: Otimização Multi-Objetivo com Pymoo**Objetivo:** Implementar um sistema completo de otimização multi-objetivo usando a biblioteca Pymoo e o algoritmo NSGA-II para resolver o Next Release Problem.**Problema:** Selecionar features para a próxima versão de um software, balanceando satisfação do cliente, custo de desenvolvimento e risco técnico.---## Configuração do Ambiente

In [ ]:
# Instalação de dependências!pip install pymoo numpy matplotlib pandas seaborn plotly scipy -q

In [ ]:
# Configurações para usar CPU e silenciar logsimport osos.environ['CUDA_VISIBLE_DEVICES'] = ''import warningswarnings.filterwarnings('ignore')# Imports principaisimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport pandas as pdimport randomfrom typing import List, Tuple, Dict, Optional, Anyfrom dataclasses import dataclassimport copy# Imports do Pymoofrom pymoo.core.problem import Problemfrom pymoo.algorithms.moo.nsga2 import NSGA2from pymoo.operators.crossover.pntx import TwoPointCrossoverfrom pymoo.operators.mutation.bitflip import BitflipMutationfrom pymoo.operators.sampling.rnd import BinaryRandomSamplingfrom pymoo.optimize import minimizefrom pymoo.indicators.hv import HVfrom pymoo.indicators.igd import IGDfrom pymoo.util.nds.non_dominated_sorting import NonDominatedSorting# Configurações de visualizaçãoplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")plt.rcParams['figure.figsize'] = (12, 8)random.seed(42)np.random.seed(42)print("✅ Ambiente configurado com sucesso!")print("🔧 Usando CPU apenas")print("📚 Bibliotecas carregadas")print(f"📦 Pymoo instalado e pronto para uso")

## Parte 1: Definição do Problema e DatasetVamos criar um dataset realista de features para o Next Release Problem com múltiplos atributos.

In [ ]:
@dataclassclass Feature:    """    Representa uma feature candidata para a próxima release.        Attributes:        nome: Nome descritivo da feature        satisfacao: Score de satisfação do cliente (1-100)        custo: Custo em horas de desenvolvimento (50-500h)        risco: Risco técnico (1-10, onde 10 é muito arriscado)        impacto_performance: Impacto na performance do sistema (-10 a +10)    """    nome: str    satisfacao: int    custo: int    risco: int    impacto_performance: int# Dataset de 20 features realistasfeatures_dataset = [    Feature("Autenticação Biométrica", 85, 320, 7, -3),    Feature("Chat em Tempo Real", 90, 450, 8, -5),    Feature("Análise Preditiva com IA", 95, 500, 9, -4),    Feature("Dashboard Customizável", 75, 180, 4, -2),    Feature("Integração com Redes Sociais", 70, 150, 3, -1),    Feature("Notificações Push Inteligentes", 80, 200, 5, -2),    Feature("Modo Offline Avançado", 85, 400, 8, 2),    Feature("Busca por Voz", 65, 280, 6, -3),    Feature("Tema Escuro/Claro", 60, 80, 2, 1),    Feature("Exportação de Relatórios PDF", 70, 120, 3, 0),    Feature("Sistema de Gamificação", 78, 350, 6, -2),    Feature("Integração com Blockchain", 55, 480, 10, -6),    Feature("Análise de Sentimento", 72, 380, 7, -3),    Feature("Reconhecimento de Imagem", 88, 420, 8, -4),    Feature("API REST Pública", 82, 280, 5, -1),    Feature("Multi-idiomas (i18n)", 68, 220, 4, 1),    Feature("Sistema de Auditoria Completo", 75, 300, 5, -2),    Feature("Cache Distribuído", 70, 350, 7, 3),    Feature("Compressão de Dados Avançada", 65, 180, 4, 2),    Feature("Backup Automático em Nuvem", 80, 250, 5, -1),]# Exibir dataset em formato tabulardf_features = pd.DataFrame([    {        'Feature': f.nome,        'Satisfação': f.satisfacao,        'Custo (h)': f.custo,        'Risco': f.risco,        'Impacto Perf.': f.impacto_performance    }    for f in features_dataset])print("\n📊 Dataset de Features para o Next Release Problem:\n")print(df_features.to_string(index=False))print(f"\n📈 Total de features candidatas: {len(features_dataset)}")print(f"💰 Custo total se todas forem implementadas: {sum(f.custo for f in features_dataset)}h")print(f"😊 Satisfação total possível: {sum(f.satisfacao for f in features_dataset)}")

## Parte 2: Formulação Multi-Objetivo com PymooVamos implementar o Next Release Problem como um problema de otimização multi-objetivo usando a estrutura do Pymoo.

In [ ]:
class NextReleaseProblem(Problem):    """    Next Release Problem formulado como otimização multi-objetivo.        Representação: Vetor binário de tamanho N (número de features)    onde 1 indica que a feature foi selecionada e 0 caso contrário.        Objetivos:        1. Maximizar Satisfação Total do Cliente        2. Minimizar Custo Total de Desenvolvimento        3. Minimizar Risco Total do Projeto        Restrições:        - Budget máximo: 2000 horas        - Número máximo de features: 12    """        def __init__(self, features: List[Feature], max_budget: int = 2000, max_features: int = 12):        """        Inicializa o problema.                Parameters:            features: Lista de features candidatas            max_budget: Orçamento máximo em horas            max_features: Número máximo de features que podem ser selecionadas        """        self.features = features        self.max_budget = max_budget        self.max_features = max_features        self.n_features = len(features)                # Definir o problema: n variáveis binárias, 3 objetivos, 2 restrições        super().__init__(            n_var=self.n_features,            n_obj=3,            n_ieq_constr=2,            xl=0,  # limite inferior (binary)            xu=1,  # limite superior (binary)            vtype=bool  # tipo de variável: booleana        )        def _evaluate(self, X, out, *args, **kwargs):        """        Avalia as soluções.                Parameters:            X: Matriz de soluções (população x n_features)            out: Dicionário para armazenar objetivos e restrições        """        # X é uma matriz onde cada linha é uma solução (indivíduo)        n_solutions = X.shape[0]                # Arrays para armazenar objetivos        f1 = np.zeros(n_solutions)  # Satisfação (maximizar -> negar para minimizar)        f2 = np.zeros(n_solutions)  # Custo (minimizar)        f3 = np.zeros(n_solutions)  # Risco (minimizar)                # Arrays para restrições        g1 = np.zeros(n_solutions)  # Budget constraint        g2 = np.zeros(n_solutions)  # Max features constraint                for i, solution in enumerate(X):            # Calcular objetivos            satisfacao_total = sum(                self.features[j].satisfacao                 for j in range(self.n_features)                 if solution[j]            )            custo_total = sum(                self.features[j].custo                 for j in range(self.n_features)                 if solution[j]            )            risco_total = sum(                self.features[j].risco                 for j in range(self.n_features)                 if solution[j]            )            n_selected = np.sum(solution)                        # Pymoo minimiza por padrão, então negamos satisfação            f1[i] = -satisfacao_total            f2[i] = custo_total            f3[i] = risco_total                        # Restrições (g <= 0 é viável)            g1[i] = custo_total - self.max_budget  # custo <= max_budget            g2[i] = n_selected - self.max_features  # n_features <= max_features                out["F"] = np.column_stack([f1, f2, f3])        out["G"] = np.column_stack([g1, g2])# Criar instância do problemaproblem = NextReleaseProblem(    features=features_dataset,    max_budget=2000,    max_features=12)print("✅ Problema NextReleaseProblem criado com sucesso!")print(f"📊 Dimensões do problema:")print(f"   - Variáveis de decisão: {problem.n_var}")print(f"   - Objetivos: {problem.n_obj}")print(f"   - Restrições: {problem.n_ieq_constr}")print(f"\n🎯 Objetivos:")print(f"   1. Maximizar Satisfação do Cliente")print(f"   2. Minimizar Custo de Desenvolvimento")print(f"   3. Minimizar Risco Técnico")print(f"\n⚠️  Restrições:")print(f"   - Budget máximo: {problem.max_budget} horas")print(f"   - Features máximas: {problem.max_features}")

## Parte 3: Configuração e Execução do NSGA-IIAgora vamos configurar e executar o algoritmo NSGA-II para resolver nosso problema multi-objetivo.

In [ ]:
# Configurar o algoritmo NSGA-IIalgorithm = NSGA2(    pop_size=100,  # Tamanho da população    sampling=BinaryRandomSampling(),  # Amostragem aleatória binária    crossover=TwoPointCrossover(prob=0.9),  # Crossover de dois pontos    mutation=BitflipMutation(prob=1.0/problem.n_var),  # Mutação bit-flip    eliminate_duplicates=True  # Eliminar soluções duplicadas)print("🔧 Algoritmo NSGA-II configurado!")print(f"   - População: {algorithm.pop_size} indivíduos")print(f"   - Crossover: Two-Point (prob=0.9)")print(f"   - Mutação: Bit-Flip (prob={1.0/problem.n_var:.4f})")print(f"\n⏳ Iniciando otimização...")# Executar a otimizaçãoresult = minimize(    problem,    algorithm,    ('n_gen', 50),  # Critério de parada: 50 gerações    seed=42,    verbose=True)print(f"\n✅ Otimização concluída!")print(f"📊 Soluções na Fronteira de Pareto: {len(result.F)}")

## Parte 4: Análise dos ResultadosVamos analisar as soluções encontradas e visualizar a Fronteira de Pareto.

In [ ]:
# Extrair soluções da fronteira de Paretopareto_solutions = result.Xpareto_objectives = result.F# Converter objetivos de volta (negamos satisfação novamente)pareto_objectives_adjusted = pareto_objectives.copy()pareto_objectives_adjusted[:, 0] = -pareto_objectives_adjusted[:, 0]  # Satisfação volta ao positivo# Criar DataFrame com os resultadosdf_pareto = pd.DataFrame({    'Solução': [f"Sol_{i+1}" for i in range(len(pareto_objectives_adjusted))],    'Satisfação': pareto_objectives_adjusted[:, 0],    'Custo (h)': pareto_objectives_adjusted[:, 1],    'Risco': pareto_objectives_adjusted[:, 2],    'N_Features': [np.sum(sol) for sol in pareto_solutions]})print("\n📊 Fronteira de Pareto - Top 10 Soluções:\n")print(df_pareto.head(10).to_string(index=False))# Estatísticas da fronteiraprint(f"\n📈 Estatísticas da Fronteira de Pareto:")print(f"   - Soluções encontradas: {len(pareto_objectives_adjusted)}")print(f"   - Satisfação: Min={df_pareto['Satisfação'].min():.0f}, Max={df_pareto['Satisfação'].max():.0f}, Média={df_pareto['Satisfação'].mean():.1f}")print(f"   - Custo: Min={df_pareto['Custo (h)'].min():.0f}, Max={df_pareto['Custo (h)'].max():.0f}, Média={df_pareto['Custo (h)'].mean():.1f}")print(f"   - Risco: Min={df_pareto['Risco'].min():.0f}, Max={df_pareto['Risco'].max():.0f}, Média={df_pareto['Risco'].mean():.1f}")print(f"   - Features: Min={df_pareto['N_Features'].min():.0f}, Max={df_pareto['N_Features'].max():.0f}, Média={df_pareto['N_Features'].mean():.1f}")

## Parte 5: Visualização da Fronteira de ParetoVamos criar visualizações 2D e 3D para entender melhor os trade-offs entre os objetivos.

In [ ]:
# Visualização 2D: Trade-offs entre pares de objetivosfig, axes = plt.subplots(1, 3, figsize=(18, 5))# Satisfação vs Custoaxes[0].scatter(    pareto_objectives_adjusted[:, 0],    pareto_objectives_adjusted[:, 1],    c=pareto_objectives_adjusted[:, 2],    cmap='viridis',    s=100,    alpha=0.6,    edgecolors='black')axes[0].set_xlabel('Satisfação do Cliente', fontsize=12, fontweight='bold')axes[0].set_ylabel('Custo (horas)', fontsize=12, fontweight='bold')axes[0].set_title('Trade-off: Satisfação vs Custo', fontsize=14, fontweight='bold')axes[0].grid(True, alpha=0.3)cbar1 = plt.colorbar(axes[0].collections[0], ax=axes[0])cbar1.set_label('Risco', fontsize=10)# Satisfação vs Riscoaxes[1].scatter(    pareto_objectives_adjusted[:, 0],    pareto_objectives_adjusted[:, 2],    c=pareto_objectives_adjusted[:, 1],    cmap='plasma',    s=100,    alpha=0.6,    edgecolors='black')axes[1].set_xlabel('Satisfação do Cliente', fontsize=12, fontweight='bold')axes[1].set_ylabel('Risco Técnico', fontsize=12, fontweight='bold')axes[1].set_title('Trade-off: Satisfação vs Risco', fontsize=14, fontweight='bold')axes[1].grid(True, alpha=0.3)cbar2 = plt.colorbar(axes[1].collections[0], ax=axes[1])cbar2.set_label('Custo (h)', fontsize=10)# Custo vs Riscoaxes[2].scatter(    pareto_objectives_adjusted[:, 1],    pareto_objectives_adjusted[:, 2],    c=pareto_objectives_adjusted[:, 0],    cmap='coolwarm',    s=100,    alpha=0.6,    edgecolors='black')axes[2].set_xlabel('Custo (horas)', fontsize=12, fontweight='bold')axes[2].set_ylabel('Risco Técnico', fontsize=12, fontweight='bold')axes[2].set_title('Trade-off: Custo vs Risco', fontsize=14, fontweight='bold')axes[2].grid(True, alpha=0.3)cbar3 = plt.colorbar(axes[2].collections[0], ax=axes[2])cbar3.set_label('Satisfação', fontsize=10)plt.tight_layout()plt.show()print("📊 Visualizações 2D criadas!")

In [ ]:
# Visualização 3D da Fronteira de Paretofrom mpl_toolkits.mplot3d import Axes3Dfig = plt.figure(figsize=(14, 10))ax = fig.add_subplot(111, projection='3d')# Plot 3Dscatter = ax.scatter(    pareto_objectives_adjusted[:, 0],    pareto_objectives_adjusted[:, 1],    pareto_objectives_adjusted[:, 2],    c=pareto_objectives_adjusted[:, 0],    cmap='viridis',    s=150,    alpha=0.7,    edgecolors='black',    linewidth=1.5)ax.set_xlabel('\nSatisfação do Cliente', fontsize=12, fontweight='bold')ax.set_ylabel('\nCusto (horas)', fontsize=12, fontweight='bold')ax.set_zlabel('\nRisco Técnico', fontsize=12, fontweight='bold')ax.set_title('Fronteira de Pareto 3D\nNext Release Problem', fontsize=16, fontweight='bold', pad=20)# Colorbarcbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.8)cbar.set_label('Satisfação', fontsize=11, fontweight='bold')ax.view_init(elev=20, azim=45)plt.tight_layout()plt.show()print("🎨 Visualização 3D da Fronteira de Pareto criada!")

## Parte 6: Soluções RepresentativasVamos selecionar e analisar algumas soluções representativas da Fronteira de Pareto que representam diferentes estratégias.

In [ ]:
# Selecionar soluções representativas# 1. Foco em Alta Satisfaçãoidx_max_satisfacao = np.argmax(pareto_objectives_adjusted[:, 0])# 2. Foco em Baixo Custoidx_min_custo = np.argmin(pareto_objectives_adjusted[:, 1])# 3. Foco em Baixo Riscoidx_min_risco = np.argmin(pareto_objectives_adjusted[:, 2])# 4. Solução Balanceada (mais próxima do centro)# Normalizar objetivos para [0, 1]norm_objs = (pareto_objectives_adjusted - pareto_objectives_adjusted.min(axis=0)) / (    pareto_objectives_adjusted.max(axis=0) - pareto_objectives_adjusted.min(axis=0))# Inverter satisfação (queremos alto) e custo/risco (queremos baixo)norm_objs[:, 0] = 1 - norm_objs[:, 0]  # Alta satisfação = 0 após normalização# Distância do ponto "ideal" (1, 0, 0) - alta satisfação, baixo custo, baixo riscoideal = np.array([0, 0, 0])distances = np.linalg.norm(norm_objs - ideal, axis=1)idx_balanceada = np.argmin(distances)# Criar função para exibir detalhes de uma soluçãodef exibir_solucao(idx, nome):    print(f"\n{'='*70}")    print(f"🎯 {nome}")    print(f"{'='*70}")        sol = pareto_solutions[idx]    objs = pareto_objectives_adjusted[idx]        print(f"\n📊 Objetivos:")    print(f"   - Satisfação do Cliente: {objs[0]:.0f}")    print(f"   - Custo de Desenvolvimento: {objs[1]:.0f}h")    print(f"   - Risco Técnico: {objs[2]:.0f}")        features_selecionadas = [        features_dataset[i].nome         for i in range(len(sol))         if sol[i]    ]        print(f"\n✅ Features Selecionadas ({len(features_selecionadas)}):")    for i, feature_nome in enumerate(features_selecionadas, 1):        feature = next(f for f in features_dataset if f.nome == feature_nome)        print(f"   {i:2d}. {feature_nome:40s} (Sat:{feature.satisfacao:2d}, Custo:{feature.custo:3d}h, Risco:{feature.risco:2d})")        return features_selecionadas# Exibir soluções representativassol1_features = exibir_solucao(idx_max_satisfacao, "Estratégia 1: FOCO EM ALTA SATISFAÇÃO")sol2_features = exibir_solucao(idx_min_custo, "Estratégia 2: FOCO EM BAIXO CUSTO")sol3_features = exibir_solucao(idx_min_risco, "Estratégia 3: FOCO EM BAIXO RISCO")sol4_features = exibir_solucao(idx_balanceada, "Estratégia 4: SOLUÇÃO BALANCEADA")print(f"\n{'='*70}")print("💡 Análise Estratégica:")print(f"{'='*70}")print("""Cada solução representa um trade-off diferente:- Estratégia 1: Maximiza satisfação, mas com custo e risco mais altos- Estratégia 2: Minimiza custo, sacrificando satisfação e aceitando algum risco- Estratégia 3: Minimiza risco, mas pode ter custo moderado- Estratégia 4: Busca equilíbrio entre todos os objetivosA escolha final depende das prioridades do negócio e contexto do projeto.""")

## Parte 7: Comparação com Abordagem Mono-ObjetivoVamos comparar a otimização multi-objetivo com abordagens mono-objetivo tradicionais usando diferentes pesos.

In [ ]:
from pymoo.algorithms.soo.nonconvex.ga import GAprint("🔄 Comparando Multi-Objetivo vs Mono-Objetivo...")print("\nExecutando otimizações mono-objetivo com diferentes pesos...\n")# Definir classe para problema mono-objetivo com pesosclass NextReleaseProblemWeighted(Problem):    """Versão mono-objetivo do problema com função de fitness ponderada."""        def __init__(self, features, max_budget, max_features, w1, w2, w3):        self.features = features        self.max_budget = max_budget        self.max_features = max_features        self.n_features = len(features)        self.w1 = w1  # peso para satisfação        self.w2 = w2  # peso para custo        self.w3 = w3  # peso para risco                super().__init__(            n_var=self.n_features,            n_obj=1,  # Apenas um objetivo            n_ieq_constr=2,            xl=0,            xu=1,            vtype=bool        )        def _evaluate(self, X, out, *args, **kwargs):        n_solutions = X.shape[0]        f = np.zeros(n_solutions)        g1 = np.zeros(n_solutions)        g2 = np.zeros(n_solutions)                for i, solution in enumerate(X):            satisfacao = sum(self.features[j].satisfacao for j in range(self.n_features) if solution[j])            custo = sum(self.features[j].custo for j in range(self.n_features) if solution[j])            risco = sum(self.features[j].risco for j in range(self.n_features) if solution[j])            n_selected = np.sum(solution)                        # Normalizar objetivos para [0, 1] para pesos terem significado            max_sat = sum(f.satisfacao for f in self.features)            max_custo = sum(f.custo for f in self.features)            max_risco = sum(f.risco for f in self.features)                        sat_norm = satisfacao / max_sat            custo_norm = custo / max_custo            risco_norm = risco / max_risco                        # Fitness ponderado (minimizar, então satisfação é negativa)            f[i] = -(self.w1 * sat_norm) + (self.w2 * custo_norm) + (self.w3 * risco_norm)                        g1[i] = custo - self.max_budget            g2[i] = n_selected - self.max_features                out["F"] = f        out["G"] = np.column_stack([g1, g2])# Configurações de pesos para testarweight_configs = [    {"nome": "Alta Satisfação", "w1": 0.7, "w2": 0.2, "w3": 0.1},    {"nome": "Baixo Custo", "w1": 0.3, "w2": 0.6, "w3": 0.1},    {"nome": "Balanceado", "w1": 0.5, "w2": 0.3, "w3": 0.2},]mono_results = []for config in weight_configs:    print(f"⚙️  Executando: {config['nome']} (w1={config['w1']}, w2={config['w2']}, w3={config['w3']})")        problem_mono = NextReleaseProblemWeighted(        features_dataset, 2000, 12,        config['w1'], config['w2'], config['w3']    )        algorithm_mono = GA(        pop_size=100,        sampling=BinaryRandomSampling(),        crossover=TwoPointCrossover(prob=0.9),        mutation=BitflipMutation(prob=1.0/problem_mono.n_var),        eliminate_duplicates=True    )        result_mono = minimize(        problem_mono,        algorithm_mono,        ('n_gen', 50),        seed=42,        verbose=False    )        # Calcular objetivos reais da melhor solução    sol = result_mono.X    satisfacao = sum(features_dataset[j].satisfacao for j in range(len(sol)) if sol[j])    custo = sum(features_dataset[j].custo for j in range(len(sol)) if sol[j])    risco = sum(features_dataset[j].risco for j in range(len(sol)) if sol[j])        mono_results.append({        'Estratégia': config['nome'],        'Satisfação': satisfacao,        'Custo': custo,        'Risco': risco,        'N_Features': np.sum(sol)    })        print(f"   ✅ Satisfação: {satisfacao:.0f}, Custo: {custo:.0f}h, Risco: {risco:.0f}\n")df_mono = pd.DataFrame(mono_results)print("\n" + "="*70)print("📊 COMPARAÇÃO: Mono-Objetivo vs Multi-Objetivo")print("="*70)print("\n🔹 Resultados Mono-Objetivo:\n")print(df_mono.to_string(index=False))print("\n🔹 Amostra da Fronteira Multi-Objetivo (5 soluções):\n")sample_idx = np.linspace(0, len(df_pareto)-1, min(5, len(df_pareto)), dtype=int)print(df_pareto.iloc[sample_idx].to_string(index=False))

In [ ]:
# Visualização comparativafig, ax = plt.subplots(figsize=(14, 8))# Plot Fronteira de Pareto (multi-objetivo)ax.scatter(    pareto_objectives_adjusted[:, 0],    pareto_objectives_adjusted[:, 1],    c='lightblue',    s=120,    alpha=0.5,    edgecolors='navy',    linewidth=1.5,    label='Fronteira de Pareto (Multi-Objetivo)',    zorder=2)# Plot soluções mono-objetivocolors = ['red', 'orange', 'green']markers = ['D', 's', '^']for i, (_, row) in enumerate(df_mono.iterrows()):    ax.scatter(        row['Satisfação'],        row['Custo'],        c=colors[i],        marker=markers[i],        s=300,        alpha=0.8,        edgecolors='black',        linewidth=2,        label=f"Mono: {row['Estratégia']}",        zorder=3    )ax.set_xlabel('Satisfação do Cliente', fontsize=14, fontweight='bold')ax.set_ylabel('Custo (horas)', fontsize=14, fontweight='bold')ax.set_title('Comparação: Multi-Objetivo vs Mono-Objetivo\nSatisfação vs Custo',              fontsize=16, fontweight='bold')ax.legend(loc='best', fontsize=11, framealpha=0.9)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n💡 Insights:")print("""1. As soluções mono-objetivo são casos especiais da fronteira multi-objetivo2. A fronteira de Pareto oferece muito mais opções de trade-off3. Mudanças nos pesos podem levar a soluções muito diferentes4. Com multi-objetivo, não precisamos decidir pesos a priori5. A fronteira expõe explicitamente os trade-offs, facilitando decisões informadas""")

## Parte 8: Métricas de Qualidade da FronteiraVamos calcular métricas para avaliar a qualidade da Fronteira de Pareto encontrada.

In [ ]:
# Calcular Hypervolume# Hypervolume mede o volume do espaço objetivo dominado pela fronteira# Maior hypervolume = melhor fronteira# Definir ponto de referência (pior caso para todos os objetivos)ref_point = np.array([    -np.min(pareto_objectives_adjusted[:, 0]) * 1.1,  # Pior satisfação (negativo)    np.max(pareto_objectives_adjusted[:, 1]) * 1.1,   # Pior custo    np.max(pareto_objectives_adjusted[:, 2]) * 1.1    # Pior risco])# Ajustar para Pymoo (satisfação precisa ser negativa)pareto_for_hv = pareto_objectives.copy()ind = HV(ref_point=ref_point)hypervolume = ind(pareto_for_hv)print("📏 Métricas de Qualidade da Fronteira de Pareto:")print(f"\n🎯 Hypervolume: {hypervolume:.2f}")print(f"   (Volume do espaço objetivo dominado pela fronteira)")# Calcular Spread (diversidade)# Spread mede quão bem distribuídas estão as soluções na fronteiradef calculate_spread(pareto_front):    """Calcula a métrica de diversidade (spread) da fronteira."""    if len(pareto_front) < 2:        return 0.0        # Normalizar objetivos    pf_norm = (pareto_front - pareto_front.min(axis=0)) / (        pareto_front.max(axis=0) - pareto_front.min(axis=0) + 1e-10    )        # Calcular distâncias entre soluções consecutivas    distances = []    for i in range(len(pf_norm) - 1):        dist = np.linalg.norm(pf_norm[i] - pf_norm[i+1])        distances.append(dist)        if len(distances) == 0:        return 0.0        d_mean = np.mean(distances)        # Spread: desvio das distâncias em relação à média    spread = np.sqrt(np.sum((np.array(distances) - d_mean) ** 2)) / (len(distances) * d_mean + 1e-10)        return spreadspread = calculate_spread(pareto_objectives_adjusted)print(f"\n📊 Spread (Diversidade): {spread:.4f}")print(f"   (Menor é melhor - indica distribuição uniforme)")# Número de soluções na fronteiran_pareto = len(pareto_objectives_adjusted)print(f"\n🔢 Número de Soluções na Fronteira: {n_pareto}")# Cobertura dos objetivosprint(f"\n📈 Cobertura dos Objetivos:")print(f"   - Satisfação: [{df_pareto['Satisfação'].min():.0f}, {df_pareto['Satisfação'].max():.0f}] (range: {df_pareto['Satisfação'].max() - df_pareto['Satisfação'].min():.0f})")print(f"   - Custo: [{df_pareto['Custo (h)'].min():.0f}, {df_pareto['Custo (h)'].max():.0f}] (range: {df_pareto['Custo (h)'].max() - df_pareto['Custo (h)'].min():.0f})")print(f"   - Risco: [{df_pareto['Risco'].min():.0f}, {df_pareto['Risco'].max():.0f}] (range: {df_pareto['Risco'].max() - df_pareto['Risco'].min():.0f})")print(f"\n✅ Métricas calculadas com sucesso!")

## Parte 9: Recomendação ExecutivaCom base na análise completa, vamos gerar uma recomendação executiva para a tomada de decisão.

In [ ]:
print("="*80)print(" " * 20 + "📋 RECOMENDAÇÃO EXECUTIVA")print("="*80)print(f"""🎯 CONTEXTO DO PROBLEMA   - Features candidatas analisadas: {len(features_dataset)}   - Budget disponível: {problem.max_budget} horas   - Limite de features: {problem.max_features}📊 RESULTADOS DA OTIMIZAÇÃO MULTI-OBJETIVO   - Soluções viáveis encontradas: {n_pareto}   - Hypervolume da fronteira: {hypervolume:.2f}   - Diversidade (Spread): {spread:.4f}🎲 SOLUÇÕES ESTRATÉGICAS RECOMENDADASBaseado na análise da Fronteira de Pareto, apresentamos 3 estratégias principais:""")# Exibir informações condensadas das 3 principais estratégiasprint("\n1️⃣  ESTRATÉGIA CONSERVADORA (Baixo Risco)")objs_conserv = pareto_objectives_adjusted[idx_min_risco]print(f"   Satisfação: {objs_conserv[0]:.0f} | Custo: {objs_conserv[1]:.0f}h | Risco: {objs_conserv[2]:.0f}")print("   Perfil: Minimiza risco técnico, ideal para projetos críticos")print("\n2️⃣  ESTRATÉGIA AGRESSIVA (Alta Satisfação)")objs_agressiva = pareto_objectives_adjusted[idx_max_satisfacao]print(f"   Satisfação: {objs_agressiva[0]:.0f} | Custo: {objs_agressiva[1]:.0f}h | Risco: {objs_agressiva[2]:.0f}")print("   Perfil: Maximiza impacto no cliente, aceita mais risco")print("\n3️⃣  ESTRATÉGIA BALANCEADA (Recomendada)")objs_balanceada = pareto_objectives_adjusted[idx_balanceada]print(f"   Satisfação: {objs_balanceada[0]:.0f} | Custo: {objs_balanceada[1]:.0f}h | Risco: {objs_balanceada[2]:.0f}")print("   Perfil: Melhor equilíbrio entre satisfação, custo e risco")print(f"""\n{'='*80}💡 ANÁLISE COMPARATIVA COM MONO-OBJETIVO{'='*80}A abordagem multi-objetivo revelou {n_pareto} soluções viáveis, enquantocada configuração mono-objetivo retorna apenas 1 solução.Vantagens da Abordagem Multi-Objetivo:✅ Expõe explicitamente os trade-offs entre objetivos✅ Não requer definição prévia de pesos (que é subjetiva)✅ Permite adaptação a mudanças de prioridades sem reprocessamento✅ Fornece "menu" de opções para tomada de decisão informada{'='*80}🎯 RECOMENDAÇÃO FINAL{'='*80}Recomendamos a ESTRATÉGIA BALANCEADA como ponto de partida, pois:- Oferece satisfação elevada ({objs_balanceada[0]:.0f} pontos)- Mantém custo controlado ({objs_balanceada[1]:.0f}h)- Risco técnico aceitável ({objs_balanceada[2]:.0f} pontos)No entanto, a decisão final deve considerar:📌 Prioridades estratégicas do negócio📌 Contexto competitivo do mercado📌 Recursos e expertise disponíveis na equipe📌 Timelines e pressões de time-to-market{'='*80}""")

## 🎓 Conclusão do WorkshopNeste laboratório prático, você:1. ✅ **Formulou um problema real** de engenharia de software (Next Release Problem) como otimização multi-objetivo2. ✅ **Implementou NSGA-II** usando a biblioteca Pymoo com configuração completa3. ✅ **Visualizou e interpretou** a Fronteira de Pareto em 2D e 3D4. ✅ **Analisou trade-offs** entre satisfação do cliente, custo e risco técnico5. ✅ **Comparou** abordagens multi-objetivo vs mono-objetivo6. ✅ **Calculou métricas** de qualidade (Hypervolume, Spread)7. ✅ **Gerou recomendações** executivas baseadas em análise quantitativa### 🚀 Próximos PassosPara aprofundar seu conhecimento:- **Experimente com mais objetivos**: Adicione "Impacto na Performance" como 4º objetivo- **Teste outros algoritmos**: NSGA-III, MOEA/D, SMS-EMOA para problemas com many-objectives- **Implemente preferências**: Use Reference Point Methods para guiar a busca- **Análise de sensibilidade**: Varie o budget e veja como a fronteira muda- **Problemas reais**: Aplique em seus próprios projetos de software### 📚 Recursos Adicionais- [Documentação Pymoo](https://pymoo.org/)- [Paper NSGA-II](https://ieeexplore.ieee.org/document/996017) (Deb et al., 2002)- [Multi-Objective Optimization Book](https://link.springer.com/book/10.1007/978-3-662-04693-1)---**🎉 Parabéns! Você completou o Workshop de Otimização Multi-Objetivo com Pymoo!**